In [ ]:
import os
import pandas as pd
import numpy as np

def save_each_row_as_npz(root_dir, ecog_folder_name, block_number, output_folder):
    """
    Process Ecog dataset and save each row of a specific block as a separate .npz file.
    
    Parameters:
        root_dir (str): Path to the root directory containing the Ecog folder.
        ecog_folder_name (str): Name of the Ecog folder.
        block_number (int): Block number to process (e.g., 1 for block1.csv).
        output_folder (str): Path to save individual row .npz files.
    """

    ecog_folder = os.path.join(root_dir, ecog_folder_name)
    if not os.path.isdir(ecog_folder):
        raise FileNotFoundError(f"Ecog folder '{ecog_folder}' not found in root '{root_dir}'.")

    os.makedirs(output_folder, exist_ok=True)
    file_count = 0

    # Loop over each Ecog subfolder (Ecog1, Ecog2, ...)
    for ecog_subfolder in sorted(os.listdir(ecog_folder)):
        if 'Ecog' not in ecog_subfolder:  
            continue
        ecog_path = os.path.join(ecog_folder, ecog_subfolder)
        if not os.path.isdir(ecog_path):
            continue

        channel_signals = []

        # Loop over 3 channels
        for ch in range(1, 4):
            channel_path = os.path.join(ecog_path, f'channel{ch}')
            block_file = os.path.join(channel_path, f'block{block_number}.csv')
            if not os.path.isfile(block_file):
                raise FileNotFoundError(f"{block_file} not found!")

            # Read CSV, no header
            df = pd.read_csv(block_file, header=None)
            signals = df.iloc[:, :1600].values  # First 1600 columns = signals
            labels = df.iloc[:, 1600].values    # Last column = label
            channel_signals.append((signals, labels))

        # Number of rows in this block
        num_rows = channel_signals[0][0].shape[0]

        # Loop over each row
        for i in range(num_rows):
            x = np.stack([ch[0][i] for ch in channel_signals], axis=0)  # (3, 1600)
            y = np.max([ch[1][i] for ch in channel_signals])            # max label

            # Save as .npz
            filename = os.path.join(output_folder, f'row_{file_count:05d}.npz')
            np.savez(filename, x=x, y=y)
            file_count += 1

    print(f"Saved {file_count} row files in '{output_folder}'.")

# =======================
# Example usage
# =======================
root_directory = '/workspace/data/brainrat_cmps364'       # Replace with your root directory
ecog_folder_name = 'Ecog'              # The folder containing Ecog1, Ecog2, ...
block_number = 1
output_folder = '/path/to/save/npz'    # Replace with where you want .npz files

save_each_row_as_npz(
    root_dir=root_directory,
    ecog_folder_name=ecog_folder_name,
    block_number=block_number,
    output_folder=output_folder
)

# To load a saved row later:
# data = np.load('/path/to/save/npz/row_00000.npz')
# X_row = data['x']   # shape (3, 1600)
# y_row = data['y']   # scalar


Saved 3203 row files in '/path/to/save/npz'.


In [9]:
import os
import numpy as np
import pandas as pd

def load_all_blocks_concatenated(root_path, base_folder):
    """
    Load all matching blocks from Ecog data folders and channels,
    and concatenate all X and Y into single arrays.
    
    Parameters:
    - root_path: str, root directory where base_folder is located
    - base_folder: str, folder containing Ecog1...Ecog6
    
    Returns:
    - X: numpy array of shape (total_rows, 3, 1600)
    - Y: numpy array of shape (total_rows,)
    """
    base_path = os.path.join(root_path, base_folder)
    all_X = []
    all_Y = []

    for ecog_folder in sorted(os.listdir(base_path)):
        if "Ecog" not in ecog_folder:
            continue
        ecog_path = os.path.join(base_path, ecog_folder)
        if not os.path.isdir(ecog_path):
            continue
        
        # Find block files in each channel
        channel_blocks = []
        for ch in range(1, 4):
            ch_folder = os.path.join(ecog_path, f'channel{ch}')
            blocks = [f for f in os.listdir(ch_folder) if f.startswith('block') and f.endswith('.csv')]
            channel_blocks.append(set(blocks))
        
        # Only keep blocks that exist in all 3 channels
        common_blocks = sorted(list(channel_blocks[0] & channel_blocks[1] & channel_blocks[2]))
        
        for block_file in common_blocks:
            channel_data = []
            labels_list = []
            for ch in range(1, 4):
                ch_folder = os.path.join(ecog_path, f'channel{ch}')
                block_path = os.path.join(ch_folder, block_file)
                df = pd.read_csv(block_path, header=None)
                signals = df.iloc[:, :-1].values

                signals = (signals - signals.mean(axis=0)) / (signals.std(axis=0) + 1e-8)
                
                labels = df.iloc[:, -1].values
                channel_data.append(signals)
                labels_list.append(labels)
            
            # Stack channel data: shape (num_rows, 3, 1600)
            channel_data = np.array(channel_data)  # (3, num_rows, 1600)
            channel_data = np.transpose(channel_data, (1, 0, 2))
            
            # Y: max label across channels
            labels_stack = np.stack(labels_list, axis=1)
            max_labels = np.max(labels_stack, axis=1)
            
            all_X.append(channel_data)
            all_Y.append(max_labels)

    # Concatenate all blocks
    X = np.vstack(all_X)
    Y = np.concatenate(all_Y)
    
    return X, Y

# Example usage:
root_path = "/workspace/data/brainrat_cmps364"
base_folder = "Ecog"

X, Y = load_all_blocks_concatenated(root_path, base_folder)
print("X shape:", X.shape)  # (total_rows, 3, 1600)
print("Y shape:", Y.shape)  # (total_rows,)


X shape: (9344, 3, 1600)
Y shape: (9344,)


There are three values: 0 (no seizure), 1 (pre-seizure), 2 (seizure). We join pre-seizure and seizure to one value because:
- they exhibit similar characteristics
- we want the shape of ECoG data to match that of EEG

In [11]:
Y[Y == 2] = 1